<a href="https://colab.research.google.com/github/runkael2/BA305-A6-Team-Project/blob/main/Rachael/NewForest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [32]:
# Libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn import tree
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import accuracy_score, confusion_matrix, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error
# used to turn categorical into dummies

In [33]:
url = "https://raw.githubusercontent.com/runkael2/BA305-A6-Team-Project/refs/heads/main/UK_Gender_Pay_Gap_workbook_allyears.csv"
paygap_df = pd.read_csv(url)

paygap_df = paygap_df.drop(columns=['Address', 'PostCode','CompanyNumber','ResponsiblePerson',
                                    'DueDate','DateSubmitted','CompanyLinkToGPGInfo',
                                    'PostCodeAbbrv', 'CurrentName'])
# getting rid of most irrelevant columns for this prediction


display(paygap_df.head(5))

,EmployerId,REGION,SicCodes,Cleaned SIC,Industry,DiffMeanHourlyPercent,DiffMedianHourlyPercent,DiffMeanBonusPercent,DiffMedianBonusPercent,MaleBonusPercent,...,FemaleLowerQuartile,MaleLowerMiddleQuartile,FemaleLowerMiddleQuartile,MaleUpperMiddleQuartile,FemaleUpperMiddleQuartile,MaleTopQuartile,FemaleTopQuartile,EmployerSize,SubmittedAfterTheDeadline,ReportingPeriod
0,676,Dorchester,85310,85310,Education,18.0,28.2,0.0,0.0,0.0,...,75.6,50.8,49.2,49.2,50.8,51.5,48.5,500 to 999,False,2017-2018
1,16879,Edinburgh,47730,47730,Wholesale & Retail Trade,2.3,-2.7,15.0,37.5,15.6,...,79.7,25.4,74.6,10.3,89.7,18.1,81.9,250 to 499,False,2017-2018
2,17677,Leeds,78300,78300,AdministrativeServices,41.0,36.0,-69.8,-157.2,50.0,...,100.0,2.0,98.0,11.0,89.0,23.0,77.0,250 to 499,True,2017-2018
3,682,Taunton,93110,93110,Arts/Entertainment,-22.0,-34.0,-47.0,-67.0,25.0,...,44.0,52.0,48.0,30.0,70.0,24.0,76.0,250 to 499,True,2017-2018
4,17101,Sunderland,"56210,\n70229","5,621,070,229",Accommodation/Food_Service,13.4,8.1,41.4,43.7,8.7,...,70.9,49.4,50.6,22.8,77.2,58.2,41.8,250 to 499,False,2017-2018


In [34]:
# Split by reporting period to evaluate predictions on later years.
period = paygap_df["ReportingPeriod"]

training_periods = ["2017-2018", "2018-2019", "2019-2020"]

train = paygap_df.loc[period.isin(training_periods)].copy()
val = paygap_df.loc[period == "2020-2021"].copy()
test = paygap_df.loc[period == "2021-2022"].copy()

print(f"Train: {len(train):,}")
print(f"Validation: {len(val):,}")
# validation used to test multiple variations before actual testing
print(f"Test: {len(test):,}")

Train: 27,625
Validation: 10,536
Test: 10,425


In [35]:
#x
features = ["EmployerSize", "Industry", "REGION"]
#y
target = "DiffMedianHourlyPercent"

In [36]:
# Turn categories into 0/1 columns
# "ignore" allows a later report to contain a category absent from training.
encoder = OneHotEncoder(handle_unknown="ignore")

In [37]:
# Learn the categories from the training years and encode those rows.
X_train = encoder.fit_transform(train[features])
y_train = train[target]

# Apply the same encoding to 2020–2021 so the columns line up.
X_val = encoder.transform(val[features])
y_val = val[target]

In [38]:
# Baseline: imagine predicting the training median for every validation row.
# MAE is the average absolute difference between actual and predicted gaps.
baseline_prediction = y_train.median()
baseline_mae = (y_val - baseline_prediction).abs().mean()

print(f"Baseline validation MAE: {baseline_mae:.2f} percentage points")

Baseline validation MAE: 12.58 percentage points


In [54]:
depth = 40
leaf_size = 2
# sets how many training reports a tree needs in each final group
rf = RandomForestRegressor(
    n_estimators=500,
    max_depth=depth,
    min_samples_leaf=leaf_size,
    random_state=0,
    criterion="squared_error",
    n_jobs=-1,
)

#trains the forest on the first three reporting periods
rf.fit(X_train, y_train)

val_mae = mean_absolute_error(y_val, rf.predict(X_val))
# predicts pay gaps for the 2020–2021 reports

print(f"Depth {depth} | Leaf size {leaf_size} | Validation MAE: {val_mae:.2f} percentage points")

Depth 40 | Leaf size 2 | Validation MAE: 10.62 percentage points


In [55]:
# The baseline predicts the training-set median gap for every employer.
baseline_prediction = y_train.median()

# Score the baseline on the same 2020–2021 reports as the forest.
baseline_val_mae = (y_val - baseline_prediction).abs().mean()
# Baseline MAE is how far off id be if i predicted the same pay gap for every employer

print(f"Baseline prediction:     {baseline_prediction:.2f}%")
print(f"Baseline validation MAE: {baseline_val_mae:.2f} percentage points")
print(f"Forest validation MAE:   {val_mae:.2f} percentage points")
print(f"Reduction in MAE:        {baseline_val_mae - val_mae:.2f} percentage points")

Baseline prediction:     9.80%
Baseline validation MAE: 12.58 percentage points
Forest validation MAE:   10.62 percentage points
Reduction in MAE:        1.97 percentage points


In [56]:
# Now, for final test


# 1. Combine the original training years with the validation year.
#    The final forest can now learn from all four earlier periods.
final_train = pd.concat([train, val], ignore_index=True)

# 2. Fit the encoder on those four years and apply it to the test year.
final_encoder = OneHotEncoder(handle_unknown="ignore")
X_final_train = final_encoder.fit_transform(final_train[features])
X_test = final_encoder.transform(test[features])

y_final_train = final_train[target]
y_test = test[target]

# 3. Refit the forest with the settings you selected on validation.
rf_final = RandomForestRegressor(
    n_estimators=500,
    max_depth=40,
    min_samples_leaf=2,
    random_state=0,
    criterion="squared_error",
    n_jobs=-1,
)
rf_final.fit(X_final_train, y_final_train)

# 4. Predict pay gaps for 2021–2022 and measure the average error.
test_predictions = rf_final.predict(X_test)
test_mae = mean_absolute_error(y_test, test_predictions)

# 5. Score the baseline on the same test reports.
#    It predicts the median gap from the four years used for the final fit.
baseline_prediction = y_final_train.median()
baseline_test_mae = (y_test - baseline_prediction).abs().mean()

print(f"Forest test MAE:   {test_mae:.2f} percentage points")
print(f"Baseline test MAE: {baseline_test_mae:.2f} percentage points")
print(f"MAE reduction:     {baseline_test_mae - test_mae:.2f} percentage points")

Forest test MAE:   10.23 percentage points
Baseline test MAE: 12.20 percentage points
MAE reduction:     1.97 percentage points
